In [9]:
# -*- coding: utf-8 -*-
"""make_production.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1neww4ldp8eCOL7D2CJarTzKPDb9TGHtt
"""
# =============================================================================
# 生産量自動入力プログラム
#
# 【このプログラムがすること】
#   1. Excelファイルの「処理対象」シートから、処理する年(A1)と月(A2)を読む。
#   2. 「2026.10」のような名前の入力シートから、消費量・閾値・発注量を読む。
#   3. 前月末在庫を決める（最初の月は手入力値、それ以外は前月の「_出力」シートから）。
#   4. 1日ずつ、生産量（発注するかどうか）と在庫量を計算する。
#   5. 入力シートをコピーして「2026.10_出力」シートを作り、計算結果を書き込む。
#   6. 元のExcelファイルに上書き保存し、部品ごとの発注回数を画面に出す。
#
# 【シートのレイアウト（入力シート・出力シート共通）】
#   2行目      : 日付（C列から月末まで）
#   4〜6行目   : 消費量（部品A, B, C）
#   8〜10行目  : 生産量（部品A, B, C）  ← プログラムが書き込む
#   12〜14行目 : 在庫量（部品A, B, C）  ← プログラムが書き込む
#   B12〜B14   : 前月末在庫
#   B17〜B19   : 閾値（部品A, B, C）
#   C17〜C19   : 発注量（部品A, B, C）
#
# 【部品の番号 i について】
#   プログラム内では部品を i = 0, 1, 2 で表す（0=部品A, 1=部品B, 2=部品C）。
#   行番号は「基準の行 + i」で計算する。
#   例：消費量の行 = 4 + i → 部品Aは4行目、部品Bは5行目、部品Cは6行目
# =============================================================================

# os : ファイルが存在するかを調べるために使う（os.path.isfile）
import os
# sys : コマンドラインの引数(sys.argv)と、エラー時の終了(sys.exit)に使う
import sys
# datetime : セルの値が「日付」かどうかを判定するために使う
from datetime import datetime

# load_workbook : openpyxl の関数。Excelファイル(.xlsx)を開いて
#                 ブック（ファイル全体）のオブジェクトを返す
from openpyxl import load_workbook


# -----------------------------------------------------------------------------
# 定数（プログラム全体で使う、変わらない値）
# -----------------------------------------------------------------------------

# 年(A1)と月(A2)が入っているシートの名前
TARGET_SHEET = "処理対象"

# 部品の名前。最後のメッセージ表示で使う。
# 並び順は Excel の行の順（部品A=4行目, 部品B=5行目, 部品C=6行目 …）と同じ。
PARTS = ["部品A", "部品B", "部品C"]

# 日付が始まる列の番号。openpyxl では A列=1, B列=2, C列=3 と数える。
FIRST_DATE_COL = 3  # C列


# -----------------------------------------------------------------------------
# AppError クラス
#
# 「利用者に見せるエラー」専用の例外（エラーの種類）。
# プログラムの途中で問題を見つけたら raise AppError("メッセージ") で処理を止める。
# 一番下の if __name__ == "__main__": の部分でこれを受け取り、
# メッセージを画面に出して終了する。
# raise した時点で処理が止まるので、その後の保存処理まで進まない
# （＝エラーのときはファイルを保存しない、という仕様を満たす）。
# -----------------------------------------------------------------------------
class AppError(Exception):
    """利用者に表示して終了するエラー"""


# -----------------------------------------------------------------------------
# is_number(v)
#
# 【役割】 v が数値（整数 int または小数 float）なら True、それ以外なら False を返す。
# 【引数】 v : セルから読んだ値（数値・文字・空欄(None)・日付など何でも来る）
# 【戻り値】 True / False
# 【注意】 Python では True/False（bool型）が int の仲間として扱われてしまうため、
#          bool は数値ではないものとして、わざわざ除外している。
#          （ExcelのTRUE/FALSEが入っていた場合に、1/0 として計算されるのを防ぐ）
# -----------------------------------------------------------------------------
def is_number(v):
    # bool は int のサブクラスなので除外する
    return isinstance(v, (int, float)) and not isinstance(v, bool)


# -----------------------------------------------------------------------------
# to_int(v)
#
# 【役割】 年(A1)・月(A2)のセルの値を整数に変換する。
# 【引数】 v : セルから読んだ値
# 【戻り値】 整数に変換できれば int、できなければ None
#   例： 10    → 10
#        10.0  → 10    （Excelでは数値が小数として読まれることがあるため）
#        10.5  → None  （小数点以下があるので年月として正しくない）
#        "10"  → None  （文字の場合は不可）
#        None  → None  （空欄）
#        True  → None  （bool は除外。理由は is_number と同じ）
# -----------------------------------------------------------------------------
def to_int(v):
    """整数ならintを返す。10.0 のような整数値のfloatもintに変換。それ以外はNone"""
    if isinstance(v, bool):
        return None
    if isinstance(v, int):
        return v
    if isinstance(v, float) and v.is_integer():
        return int(v)
    return None


# -----------------------------------------------------------------------------
# read_number(ws, row, col)
#
# 【役割】 シートの指定したセルから数値を読む。
#          数値でなければ（空欄・文字など）その場でエラーにして止める。
#          閾値・発注量・前月末在庫を読むときに使う。
# 【引数】 ws  : 読むシート（ワークシートのオブジェクト）
#          row : 行番号（1始まり）
#          col : 列番号（1始まり。A=1, B=2, C=3 …）
# 【戻り値】 セルの数値
# 【エラー】 「数値でない値があります：2026.10!B17」のように、
#            シート名とセル番地を付けて表示する（外部仕様書5.2）。
# -----------------------------------------------------------------------------
def read_number(ws, row, col):
    """数値セルを読む。空欄・数値以外ならエラー（外部仕様書5.2「数値でない値」）"""
    # cell : 指定したセルのオブジェクト。.value で値、.coordinate で "B17" のような番地が取れる
    cell = ws.cell(row=row, column=col)
    if not is_number(cell.value):
        raise AppError(f"数値でない値があります：{ws.title}!{cell.coordinate}")
    return cell.value


# -----------------------------------------------------------------------------
# date_columns(ws)
#
# 【役割】 2行目に日付が入っている列の番号を、C列から順に集めて返す。
#          月によって日数（列の数）が違うので、毎回シートを見て数える。
# 【引数】 ws : 調べるシート
# 【戻り値】 列番号のリスト
#   例：10月（31日）なら [3, 4, 5, …, 33]（C列〜AG列）
#       9月（30日）なら  [3, 4, 5, …, 32]（C列〜AF列）
# 【しくみ】 C列から右へ1列ずつ見ていき、2行目が日付でなくなった所で止める。
# 【注意】 ws.max_column（シートの最後の列）は、書式だけ設定された空のセルも
#          数えてしまうことがあるので使わない。
# -----------------------------------------------------------------------------
def date_columns(ws):
    """C列から右へ、2行目が日付である間の列番号リスト"""
    # cols : 見つかった列番号をためていくリスト
    cols = []
    # c : いま調べている列の番号。C列(3)から始める
    c = FIRST_DATE_COL
    while isinstance(ws.cell(row=2, column=c).value, datetime):
        cols.append(c)
        c += 1
    return cols


# -----------------------------------------------------------------------------
# main(argv)
#
# 【役割】 プログラムの本体。仕様書（内部仕様書）の 1〜11 の順に処理する。
# 【引数】 argv : コマンドライン引数のリスト。
#                 argv[0] はプログラム自身の名前、argv[1] が Excel ファイルのパス。
#                 例： ["make_production.py", "生産計画自動計算.xlsx"]
# 【戻り値】 なし。問題があれば AppError を raise して止まる。
# -----------------------------------------------------------------------------
def main(argv):
    # =========================================================================
    # 1. 引数とファイルのチェック
    # =========================================================================
    # 引数が「プログラム名＋ファイル名」の2つでなければエラー
    if len(argv) != 2:
        raise AppError("Excelファイルを1つ指定してください")
    # path : 処理する Excel ファイルのパス
    path = argv[1]
    # そのパスにファイルが存在しなければエラー
    if not os.path.isfile(path):
        raise AppError(f"ファイルが見つかりません：{path}")

    # =========================================================================
    # 2. ブックを開く
    # =========================================================================
    # wb : Excel ファイル全体（ブック）。wb["シート名"] で各シートを取り出せる。
    # 【注意】load_workbook(path, data_only=True) にしてはいけない。
    #   data_only=True だと数式の代わりに「計算結果の値」を読み込むため、
    #   そのまま保存すると入力シートの数式（=B12-C4+C8 など）が消えてしまう。
    try:
        wb = load_workbook(path)
    except Exception:
        # .xlsx 以外のファイルや壊れたファイルを指定した場合
        raise AppError(f"Excelファイルを開けません：{path}")

    # =========================================================================
    # 3. 処理対象の年月を読む
    # =========================================================================
    # wb.sheetnames : ブック内の全シート名のリスト
    if TARGET_SHEET not in wb.sheetnames:
        raise AppError("「処理対象」シートがありません")
    # ws_target : 「処理対象」シート
    ws_target = wb[TARGET_SHEET]
    # y : 処理する年（A1）、m : 処理する月（A2）。整数でなければ None になる
    y = to_int(ws_target["A1"].value)
    m = to_int(ws_target["A2"].value)
    # 年は4桁（1000〜9999）、月は1〜12でなければエラー
    if y is None or m is None or not (1000 <= y <= 9999) or not (1 <= m <= 12):
        raise AppError("処理対象の年月が正しくありません")

    # name : 入力シートの名前。月の前に0は付けない（例：2026.9、2026.10）
    name = f"{y}.{m}"
    # out_name : これから作る出力シートの名前（例：2026.10_出力）
    out_name = name + "_出力"
    if name not in wb.sheetnames:
        raise AppError(f"シート「{name}」がありません")
    # ws_in : 当月の入力シート。値を読むだけで、書き込みはしない
    ws_in = wb[name]

    # prev_y, prev_m : 前月の年と月。
    #   1月なら前年の12月（例：2027.1 → 2026.12）、それ以外は同じ年の前の月。
    #   「A if 条件 else B」は、条件が正しければA、そうでなければBになる書き方。
    if m == 1:
        prev_y = y - 1
        prev_m = 12
    else:
        prev_y = y
        prev_m = m - 1
    # prev_name : 前月の入力シート名（例：2026.9）
    prev_name = f"{prev_y}.{prev_m}"
    # prev_out_name : 前月の出力シート名（例：2026.9_出力）
    prev_out_name = prev_name + "_出力"

    # =========================================================================
    # 4. 日付の列を求める
    # =========================================================================
    # cols : 当月の日付が入っている列番号のリスト（1日〜月末）
    cols = date_columns(ws_in)
    # 日付が1つも見つからない場合は、シートの作りがおかしいのでエラー
    if not cols:
        raise AppError(f"シート「{name}」の2行目に日付がありません")

    # =========================================================================
    # 5. 前月末在庫を決める
    #    pre_stock : 前月末在庫のリスト [部品A, 部品B, 部品C]
    #    from_prev : 前月の出力シートから取ってきたら True、手入力値なら False。
    #                True のときだけ、後で出力シートの B12〜B14 に書き込む。
    # =========================================================================
    if prev_name not in wb.sheetnames:
        # ---- 前月の入力シートがない ＝ 最初の月：B12〜B14 の手入力値を使う ----
        # まず空欄チェック（空欄なら専用のメッセージを出す）
        for i in range(3):
            if ws_in.cell(row=12 + i, column=2).value is None:
                raise AppError("前月末在庫（B12〜B14）を入力してください")
        # 次に数値チェックをしながら読む（文字が入っていたらエラー）
        # [ … for i in range(3)] は、i = 0, 1, 2 について計算した結果をリストにする書き方
        pre_stock = [read_number(ws_in, 12 + i, 2) for i in range(3)]
        from_prev = False
    else:
        # ---- 前月の入力シートがある：前月の「_出力」シートの最終日の在庫を使う ----
        # 前月の出力シートがない ＝ 前月をまだ実行していないのでエラー
        if prev_out_name not in wb.sheetnames:
            raise AppError(
                f"シート「{prev_out_name}」がありません。前月を先に実行してください"
            )
        # ws_prev : 前月の出力シート（読むだけで、書き込みはしない）
        ws_prev = wb[prev_out_name]
        # prev_cols : 前月の日付が入っている列番号のリスト
        prev_cols = date_columns(ws_prev)
        if not prev_cols:
            raise AppError(f"シート「{prev_out_name}」の2行目に日付がありません")
        # last_col : 前月の最終日の列番号（リストの最後の要素 [-1]）
        last_col = prev_cols[-1]
        # 前月最終日の在庫（12〜14行目）を読む
        pre_stock = [read_number(ws_prev, 12 + i, last_col) for i in range(3)]
        from_prev = True

    # =========================================================================
    # 6. 閾値・発注量を読む（当月の入力シートから）
    # =========================================================================
    # threshold : 閾値のリスト [部品A, 部品B, 部品C]（B17, B18, B19）
    threshold = [read_number(ws_in, 17 + i, 2) for i in range(3)]
    # order_qty : 発注量のリスト [部品A, 部品B, 部品C]（C17, C18, C19）
    order_qty = [read_number(ws_in, 17 + i, 3) for i in range(3)]

    # =========================================================================
    # 7. 計算（入力シートには書き込まず、結果はリストに保持する）
    # =========================================================================
    # prod_list  : 生産量の計算結果。prod_list[部品番号][列番号] = 生産量
    # stock_list : 在庫量の計算結果。stock_list[部品番号][列番号] = 在庫量
    #   どちらも「部品3つ分の辞書(dict)」のリスト。
    #   例：prod_list[0][3] は「部品A の C列（1日）の生産量」
    prod_list = [{} for _ in range(3)]
    stock_list = [{} for _ in range(3)]
    # count : 部品ごとの発注回数 [部品A, 部品B, 部品C]（最後のメッセージで使う）
    count = [0, 0, 0]

    # 外側のループ：部品 i = 0(A), 1(B), 2(C) の順に1つずつ処理する
    for i in range(3):
        # prev : 「前日の在庫」。1日目の前日在庫は前月末在庫から始める
        prev = pre_stock[i]

        # 内側のループ：1日目の列から月末の列まで、1日ずつ処理する
        for c in cols:
            # ---- (1) 当日の消費量を読む ----
            # cons_cell : 消費量のセル（部品Aなら4行目、Bなら5行目、Cなら6行目）
            cons_cell = ws_in.cell(row=4 + i, column=c)
            # 空欄や文字ならエラー（消費量は全日入力必須。土日は0）
            if not is_number(cons_cell.value):
                raise AppError(f"消費量の値が異常です：{name}!{cons_cell.coordinate}")
            # cons : 当日の消費量
            cons = cons_cell.value

            # ---- (2) 当日の生産量を決める ----
            # 当日の在庫を計算する「前」に、前日の在庫(prev)で判定するのがポイント。
            # d : 当日の日付（2行目の値）
            d = ws_in.cell(row=2, column=c).value
            # d.weekday() : 曜日を数字で返す（月=0, 火=1, 水=2, 木=3, 金=4, 土=5, 日=6）
            #   → 5未満なら月〜金（平日）。祝日は区別しないので平日扱いになる。
            # 「平日」かつ「前日の在庫が閾値以下」なら、発注量だけ生産する
            if d.weekday() < 5 and prev <= threshold[i]:
                # prod : 当日の生産量
                prod = order_qty[i]
                # 発注したので回数を1増やす
                count[i] += 1
            else:
                # 発注しない日は 0
                prod = 0

            # ---- (3) 当日の在庫を計算する ----
            # stock : 当日の在庫 = 前日在庫 − 消費量 + 生産量
            stock = prev - cons + prod

            # ---- (4) 計算結果を保持する（シートへの書き込みは 9. でまとめて行う）----
            prod_list[i][c] = prod
            stock_list[i][c] = stock

            # ---- (5) 当日の在庫を、翌日にとっての「前日の在庫」にする ----
            prev = stock

    # =========================================================================
    # 8. 出力シートを作る
    # =========================================================================
    # 同じ名前の出力シートがすでにあれば削除する（作り直すため）
    if out_name in wb.sheetnames:
        wb.remove(wb[out_name])
    # ws_out : 出力シート。入力シートを丸ごとコピーして作る
    #   （copy_worksheet で作ったシートは、ブックの一番最後に追加される）
    ws_out = wb.copy_worksheet(ws_in)
    # コピーしたシートの名前を「YYYY.M_出力」に変える
    ws_out.title = out_name
    # in_idx  : 入力シートが左から何番目か（0始まり）
    in_idx = wb.sheetnames.index(name)
    # out_idx : 出力シートが左から何番目か（今は一番最後）
    out_idx = wb.sheetnames.index(out_name)
    # 出力シートを、入力シートのすぐ後ろ（in_idx + 1 番目）へ移動する。
    #   offset は「今の位置から何個ずらすか」。マイナスなら左（前）へ動く。
    #   例：入力シートが2番目、出力シートが6番目なら、offset = (2+1) - 6 = -3
    wb.move_sheet(ws_out, offset=(in_idx + 1) - out_idx)

    # =========================================================================
    # 9. 出力シートに書き込む
    # =========================================================================
    for i in range(3):
        for c in cols:
            # 生産量（8〜10行目）：0 または発注量
            ws_out.cell(row=8 + i, column=c).value = prod_list[i][c]
            # 在庫量（12〜14行目）：計算した数値。コピー元の数式は上書きされて消える
            ws_out.cell(row=12 + i, column=c).value = stock_list[i][c]
    # 前月の出力シートから在庫を取ってきた場合だけ、B12〜B14 に書き込む
    # （最初の月は手入力値がコピーされているので、そのままにする）
    if from_prev:
        for i in range(3):
            ws_out.cell(row=12 + i, column=2).value = pre_stock[i]

    # =========================================================================
    # 10. 保存（元のファイルに上書き）
    # =========================================================================
    try:
        wb.save(path)
    except PermissionError:
        # Excel でファイルを開いたままだと、Windows では書き込みが拒否される
        raise AppError("保存できません。Excelでファイルを閉じてから実行してください")

    # =========================================================================
    # 11. 完了メッセージを表示する
    # =========================================================================
    print(f"{out_name} を作成しました。")
    # enumerate(PARTS) は、(0, "部品A"), (1, "部品B"), (2, "部品C") の順に取り出す
    for i, part in enumerate(PARTS):
        print(f"  {part}：発注 {count[i]}回")


# =============================================================================
# プログラムの入口
#
# このファイルを直接実行したときだけ、下の部分が動く
# （他のプログラムから import されたときは動かない）。
# main() の中で AppError が起きたら、メッセージを表示して終了コード1で終わる。
# =============================================================================
if __name__ == "__main__":
    try:
        # Modified: Pass the Excel file path as an argument to main
        # 【注意】Colab ではコマンドライン引数を渡せないため、ファイルのパスを
        #   ここに直接書いている。Windows のコマンドプロンプトで
        #   「python make_production.py 生産計画自動計算.xlsx」のように実行する場合は、
        #   次の行を  main(sys.argv)  に戻すこと（外部仕様書の実行方法と同じになる）。
        main([sys.argv[0], "/content/sample_data/生産計画自動計算.xlsx"])
    except AppError as e:
        # e : 発生した AppError。print すると raise したときのメッセージが出る
        print(e)
        # 終了コード 1 = 異常終了（0 が正常終了）
        sys.exit(1)


2026.10_出力 を作成しました。
  部品A：発注 7回
  部品B：発注 8回
  部品C：発注 12回
